<a href="https://colab.research.google.com/github/jollyoli93/KnowYourLoadExactly/blob/main/Crane_Detector_YOLOv8_Full_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Starter Code

In [ ]:
!python --version

In [ ]:
from google.colab import userdata


In [ ]:
!pip install roboflow
!pip install pyyaml

!pip install roboflow

roboapi = userdata.get('roboflow')

from roboflow import Roboflow
rf = Roboflow(api_key=roboapi)
project = rf.workspace("masterproject-mvazq").project("test_only")
version = project.version(2)
dataset = version.download("yolov8")

# """ full sized imgs"""

# from roboflow import Roboflow
# rf = Roboflow(api_key="roboflow")
# project = rf.workspace("masterproject-mvazq").project("test_only")
# version = project.version(1)
# dataset = version.download("yolov8")

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install ultralytics==8.3.190

In [ ]:
# !pip install --upgrade onnx onnxscript

In [ ]:
# !pip install wandb

In [ ]:
# from wandb.integration.ultralytics import add_wandb_callback
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
# import cv2
import numpy as np
import torch

In [ ]:
def extract_coords(coords):
    coord_split = [
        [float(x) for x in line.strip().split()]
        for line in coords
        if line.strip()
    ]
    return coord_split


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
from pathlib import Path

from math import sqrt

#Gets statistics from training data for MLP ROI Predictor
def get_pairs(coords, from_file=True):

  coord_split = []

  if from_file:
    coord_split = extract_coords(coords)
  else:
    coord_split = coords

  hooks = [d for d in coord_split if d[0] == 0]  # hook
  loads = [d for d in coord_split if d[0] == 1]  # load

  pairs = {'hook': [], 'load': []}
  used_hooks = set()

  for load in loads:
      xc_load, yc_load = load[1], load[2]
      h_load = load[-1]
      closest = None
      closest_idx = -1
      min_dist = float("inf")

      for i, hook in enumerate(hooks):
          if i in used_hooks:
              continue

          xc_hook, yc_hook = hook[1], hook[2]

          yt_load = yc_load - (h_load /2)
          yb_hook = yc_hook + hook[-1]/2 #Location at bottom of hook

          # Hook must be above load
          if yb_hook > yt_load:
              continue

          # Horizontal closeness
          dist = abs(xc_load - xc_hook)
          if dist < min_dist:
              min_dist = dist
              closest = hook
              closest_idx = i

      if closest is not None:
          xc, yc, w, h = closest[1:5]
          # aspect = w / h if h != 0 else 0.0
          # area = w*h
          yb = yc + (h / 2)
          #dist = sqrt((xc - xc_load)**2 + (yc - yc_load)**2). feading load dims to features might create shortcut/memorisations

          # # get relative offsets normalised
          dx = (xc_load - xc) /w
          dy = (yc_load - yb) /h

          pairs['hook'].append(
              [xc, yc, w, h] + [yb]
          )
          pairs['load'].append([dx, dy])

          used_hooks.add(closest_idx)

  return pairs

In [ ]:
train_labels = "/content/drive/MyDrive/Crane-Detector/YOLOv8 Normal/K_Fold/train/2026-01-26_5-Fold_Cross-val/split_4/train/labels"

# 1. Collect all data for fitting
train_feats = []
train_targets = []

for label_file in Path(train_labels).glob("*.txt"):
    with open(label_file) as f:
        pairs = get_pairs(f.readlines())
        if pairs:
            train_feats.extend(pairs['hook'])
            train_targets.extend(pairs['load'])

# 2. Fit the scalers
feat_scaler = StandardScaler().fit(train_feats)
target_scaler = StandardScaler().fit(train_targets)

In [ ]:
def crop_load(img, x, y, w, h):
  """
    Takes as input, top left corner, width and height.
  """
  crop_img = img.copy()
  cropped_img = crop_img[y:y+h,x:x+w]

  return cropped_img


In [ ]:
def box_gating(boxes):
  """
    Boxes contain a list of [class_idx, xc, yc, w, h, conf, src, hook_id] in pixel space.
    class 0 = hook, class 1 = load
    src in {"hook","obj","roi"}
    hook_id is an int for hook/roi loads, None for obj loads
    Returns loads only
  """
  if not boxes:
    print("Empty boxes")
    return []

  hooks = [d for d in boxes if d[6] == "hook"]
  obj_loads = [d for d in boxes if d[6] == "obj"]
  roi_loads  = [d for d in boxes if d[6] == "roi"]

  used_obj = set()
  matched_obj = set()
  kept_loads = []

  # For each hook find its highest matching load
  for hook in hooks:
    hook_x, hook_y, hook_w = hook[1], hook[2], hook[3]
    hook_id = hook[7]
    hook_boundary = hook_w * 3.0

    best_obj_idx = None
    best_obj_conf = float("-inf")

    for i, load in enumerate(obj_loads):
      load_x, load_y = load[1], load[2]

      # Gating - if it doesnt match then assume its on another hook
      if load_y < hook_y:
        continue
      if load_x < (hook_x - hook_boundary) or load_x > (hook_x + hook_boundary):
        continue

      matched_obj.add(i)

      if i in used_obj:
        continue

      conf = load[5]
      if conf > best_obj_conf:
        best_obj_conf = conf
        best_obj_idx = i

    if best_obj_idx is not None:
      kept_loads.append(obj_loads[best_obj_idx])
      used_obj.add(best_obj_idx)
    else:
      roi_candidate = [d for d in roi_loads if d[7] == hook_id]
      if roi_candidate:
        kept_loads.append(roi_candidate[0])

  for i, load in enumerate(obj_loads):
    if i not in matched_obj:
      kept_loads.append(load)

  return kept_loads

In [ ]:
def corner_to_center(xt, yt, w,h):
  xc = xt+(w/2)
  yc = yt+(h/2)
  return xc, yc, w, h

In [ ]:
def center_to_corners(xc, yc, w,h):
  xl = xc - (w/2)
  xr = xc + (w/2)
  yt = yc - (h/2)
  yb = yc + (h/2)
  return xl, xr, yt, yb, w, h

#One Cycle

In [ ]:
%cd /content/drive/MyDrive/Crane-Detector

## Load YOLO Model

In [ ]:
best_weights = "/content/drive/MyDrive/Crane-Detector/runs/detect/best_260126_train7/weights/best.pt"

In [ ]:
yolo_model = YOLO(best_weights)

In [ ]:
test_path = "/content/drive/MyDrive/Crane-Detector/First Experiment/ALL_2CLS_NOAUG/test/images"

In [ ]:
# from pathlib import Path

# results = []

# for img in Path(test_path).glob("*jpg"):
#   results.append(model.predict(img, stream=False))

## Run Image

In [ ]:
from google.colab.patches import cv2_imshow
import cv2

# img = cv2.imread("/content/drive/MyDrive/Crane-Detector/TEST_ONLY-1/test/images/2503061230060000_jpg.rf.96a86533e4d968fcc59cc8d2f078560d.jpg") # 4000w, 6000h SLOW!

# img = cv2.imread('/content/drive/MyDrive/Crane-Detector/First Experiment/All_2LCS_AUG/valid/images/2502240810060000_jpg.rf.56156243b1ad02bee4d57d2e65850cca.jpg') #1 hook, 1 load
# img = cv2.imread('/content/drive/MyDrive/Crane-Detector/First Experiment/All_2LCS_AUG/test/images/2505301040060000_jpg.rf.e1ed3ff61e40b48bd7aecdf5f7df6050.jpg') #1 hook, 1 load
img = cv2.imread('/content/drive/MyDrive/Crane-Detector/First Experiment/All_2LCS_AUG/test/images/2505301210060000_jpg.rf.4e2d4447ae977df17c79484f248ebb27.jpg') # 2 hook
# img = cv2.imread('/content/drive/MyDrive/Crane-Detector/First Experiment/All_2LCS_AUG/test/images/2505081520060000_jpg.rf.0d654dfb0aa7f32b6b1385e4ae5c7e5f.jpg')  #1 hook, 1 load (multiple loads matching)
# img = cv2.imread('/content/drive/MyDrive/Crane-Detector/First Experiment/All_2LCS_AUG/test/images/2505081740060000_jpg.rf.f446e77de4d540793a42f8ff9e3e8cda.jpg')  #1 hook, 1 load real
# img = cv2.imread('/content/drive/MyDrive/Crane-Detector/First Experiment/All_2LCS_AUG/test/images/2507190940060000_jpg.rf.25413f580b6dab2700fab296e0eca813.jpg') #1 hook, 1 load


cv2_imshow(img)

In [ ]:
img.shape

## Load Regression Model

In [ ]:
import torch

class LinearRegression(torch.nn.Module):
    def __init__(self, in_feat, out_feat, hidden) -> None:
        super().__init__()
        self.linear = torch.nn.Sequential(
            torch.nn.Linear(in_feat, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, out_feat)
        )

    def forward(self, x):
        x = self.linear(x)
        return x

In [ ]:
PATH = "/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/LoadRegressionStandardv2.pth"
cropPredict = LinearRegression(5, 2, 64)
cropPredict.load_state_dict(torch.load(PATH, weights_only=True, map_location=torch.device('cpu')))
cropPredict.eval()

In [ ]:
import numpy as np
import torch

def predict_load_roi(img_size, hook_coords):
    """
    hook_coords: normalized YOLO (x_center, y_center, w, h) in [0,1]
    img_size: (img_height, img_width) or (H, W, C) - we use first two.
    dims: optional crop size control
    """

    # image dims
    img_height = int(img_size[0])
    img_width  = int(img_size[1])

    hook_coords_np = np.asarray(hook_coords, dtype=np.float32).reshape(-1)

    x, y, w, h = hook_coords_np  # normalized
    yb_hook = y + (h / 2.0)

    # features for scaler: [x, y, w, h, yb_hook]
    hook_features = np.array([[x, y, w, h, yb_hook]], dtype=np.float32)
    xs_scaled = feat_scaler.transform(hook_features)  # numpy (1, 5)

    # Get predictions
    xs_t = torch.from_numpy(xs_scaled).float() #.to(device)
    cropPredict.eval()
    with torch.no_grad():
        pred_scaled = cropPredict(xs_t).cpu().numpy()

      # pred_inv is [dx, dy]
    pred_inv = target_scaler.inverse_transform(pred_scaled.reshape(1, -1))
    print(pred_inv)
    # dx, dy = pred_inv[0]
    dx, dy = float(pred_inv[0, 0]), float(pred_inv[0, 1])

    # ---- apply offsets relative to hook size (still normalized [0-1]) ----
    x_pred = x + (dx * w)
    y_pred = yb_hook + (dy * h)

    xc = int(round(x_pred * img_width ))
    yc = int(round(y_pred * img_height))

    #crop to top of hook
    gap_pixels = yc - (yb_hook * img_height)

    crop_h = int(gap_pixels * 1.6)
    crop_w = int(crop_h * 1.2)      # Usually loads are wider than they are tall

    xt = int(xc - (crop_w // 2))
    yt = int(yc - (crop_h // 2))

    # Clamp to image bounds
    xt = max(0, min(xt, img_width - crop_w))
    yt = max(0, min(yt, img_height - crop_h))

    return xt, yt, crop_w, crop_h


## Load Classifier Model

In [ ]:
# %cd /content/drive/MyDrive/Crane-Detector/Image_Classes/

In [ ]:
import torch
import albumentations as A
import numpy as np

checkpoint = torch.load('/content/drive/MyDrive/Crane-Detector/Image_Classes/crane_classifier.pth', map_location='cpu', weights_only=False)

class_model = checkpoint['model']
class_model.eval()

if torch.cuda.is_available():
    class_model = class_model.cuda()

cls_names = checkpoint['cls_names']
cls_index = checkpoint['cls_index']

print("Model loaded successfully!")

In [ ]:
val_transforms = A.Compose([
    A.Resize(height=240, width=240, p=1.0),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0),
])

In [ ]:
def predict_one_image(model, image_array, transforms=val_transforms):
    """
    Predict class for a single image from test set
    """
    model.eval()

    # Apply transforms
    transformed = transforms(image=image_array)['image']
    x = np.transpose(transformed, (2, 0, 1))
    x = torch.tensor(x, dtype=torch.float32).unsqueeze(0)  # Add batch dimension

    # Move to GPU if available
    if torch.cuda.is_available():
        x = x.cuda()

    with torch.no_grad():
        output = model(x)
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax(dim=1).item()
        confidence = probs[0, pred_idx].item()

    pred_class = cls_names[pred_idx]

    return pred_class, confidence


In [ ]:
# # Test prediction
# predicted_class, confidence = predict_one_image(class_model, img)

# print(f"Predicted Class: {predicted_class}")
# print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")

## One Cycle - Crane Load Predictor

In [ ]:
def second_stage_detector(crop, img_size):
  print("Predicting Second Crop")
  results = yolo_model.predict(crop, imgsz=img_size, stream=False)

  for result in results:
    if not result.boxes:
      continue

    for box in result.boxes.cpu():
        class_id = int(box.cls[0])
        class_name = yolo_model.names[class_id]  # Get class name using the class ID
        conf = float(box.conf[0])
        print(f"Class: {class_name}")  # Print class name and box coordinates
        label = f"{class_name}: {conf:.2f}"

        if class_id == 1: #Load Class
        #  LOCAL coordinates (relative to the ROI)
          lx1, ly1, lx2, ly2 = box.xyxy[0].cpu().numpy().astype(int)
          print(lx1, ly1, lx2, ly2)

          w, h = lx2 - lx1, ly2 - ly1
          print("w,h: ", w, " " , h)
          # actual cropped image of the load
          load_only_crop = crop[ly1:ly2, lx1:lx2]
          return load_only_crop, lx1, ly1, lx2, ly2, conf
  return None, 0,0,0,0, 0

In [ ]:
def predict_from_hook(image, box_size, dims,class_model):
    pts = predict_load_roi(box_size, dims)
    xt, yt, crop_w, crop_h = pts
    print(xt, yt)

    # extract the ROI from the main image
    ROI = image[yt:yt+crop_h, xt:xt+crop_w]
    if ROI.size == 0: return

    cv2.rectangle(img, (xt, yt), (xt + crop_w, yt + crop_h), (0, 255, 0), 2)
    cv2.putText(img, f"ROI", (xt, yt - 10),
      cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)

    # Find the load inside that ROI
    load_crop, lx1, ly1, lx2, ly2, yolo_conf = second_stage_detector(ROI, max(crop_w, crop_h))

    if load_crop is not None:
        # GLOBAL COORDINATES: ROI start + Local YOLO start
        gx1, gy1 = xt + lx1, yt + ly1
        gx2, gy2 = xt + lx2, yt + ly2
        w = gx2 - gx1
        h = gy1 - gy2

        # Draw the final accurate box
        predicted_class, confidence = predict_one_image(class_model, load_crop)
        cv2.rectangle(img, (gx1, gy1), (gx2, gy2), (0, 255, 0), 3)
        cv2.putText(img, f"Predicted 2 stage Crop: {predicted_class}, Conf: {confidence:.2f}", (gx1, gy1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        return gx1, gy1, h ,w, confidence
    else:
        print("MLP proposed a region, but YOLO stage 2 found no load there.")
        print("Predicting from ROI")
        print()

        predicted_class, confidence = predict_one_image(class_model, ROI)
        print(f"Predicted ROI Class: {predicted_class}")
        print(f"Confidence of ROI: {confidence:.4f} ({confidence*100:.2f}%)")

        cv2.rectangle(img, r[:2], r[2:], (0, 255, 0), 2)
        cv2.putText(img, f"{predicted_class}: {confidence:.2f}", (xt, yt - 10),
          cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)
        # returns corner point and crop w, h
        return xt, yt, crop_w, crop_h, confidence

In [ ]:
# Run prediction on the image
img = cv2.imread("/content/TEST_ONLY-1/test/images/2503080930060000_jpg.rf.57d88f19a00620fedbf013c340c48e0a.jpg")
results = yolo_model.predict(img, imgsz=1504, stream=False)

In [ ]:
# Iterate over the results

for result in results:
    boxes = result.boxes.cpu()  # Get boxes on CPU in numpy format

    box_preds = []

    for idx, box in enumerate(boxes):  # Iterate over boxes
        r = box.xyxy[0].cpu().numpy().astype(int) # Get corner points as int (Actual Co-ords)

        class_id = int(box.cls[0])  # Get class ID
        class_name = yolo_model.names[class_id]  # Get class name using the class ID
        conf = float(box.conf[0])
        print(f"Class: {class_name}, Box: {r}")  # Print class name and box coordinates
        label = f"{class_name}: {conf:.2f}"

        # Get predicted co-ordinates if not load detected

        if class_id == 1: #Load Class
          xt, yt, xb, yb = r.tolist()
          w, h = box.xywh[0].cpu().numpy().astype(int)[2:4]

          w, h = int(w), int(h)

          crop = crop_load(img, xt,yt,w,h)
          cv2_imshow(crop)

          #classify
          predicted_class, confidence = predict_one_image(class_model, crop)
          print(f"Predicted Original Class: {predicted_class}")
          print(f"Confidence of Original: {confidence:.4f} ({confidence*100:.2f}%)")

          cv2.rectangle(img, r[:2], r[2:], (0, 255, 0), 2)
          cv2.putText(img, f"{predicted_class}: {confidence:.2f}", (xt, yt - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)

          # Append to box_preds for gating
          center = corner_to_center(xt, yt, w, h)
          if isinstance(center, np.ndarray):
              center = center.tolist()
          center = [float(center[0]), float(center[1]), float(center[2]), float(center[3])]

          # use OD conf not class for gating
          box_preds.append([1, *center, conf, "obj", None])

        else:
          hook_dims_n = box.xywhn[0]
          hook_dims = box.xywh[0].cpu().tolist()
          print("Hook found - Crop ROI")
          xt,yt, w,h, confidence = predict_from_hook(img, box.orig_shape, hook_dims_n, class_model)
          print("hook load",xt,yt, w,h, " Center:", corner_to_center(xt, yt, w, h))

          # Append to box_preds for gating
          center = corner_to_center(xt, yt, w, h)

          if isinstance(center, np.ndarray):
              center = center.tolist()
          center = [float(center[0]), float(center[1]), float(center[2]), float(center[3])]
          box_preds.append([0, *hook_dims, None, "hook", idx])
          box_preds.append([1, *center, confidence, "roi", idx])

    print(box_preds)
    gated_boxes = box_gating(box_preds)

In [ ]:
cv2_imshow(img)

# Know Your Load Exactly

## ROI Predict Class


In [ ]:
class LinearRegression(torch.nn.Module):
    def __init__(self, in_feat, out_feat, hidden):
        super().__init__()
        self.linear = torch.nn.Sequential(
            torch.nn.Linear(in_feat, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, out_feat),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        return self.linear(x)

class Roi_Predict:
  def __init__(self, model_path):
    self.model = LinearRegression(5, 2, 64)
    state = torch.load(model_path,
                        map_location="cpu",
                        weights_only=True)
    self.model.load_state_dict(state)
    self.model.eval()

  def predict_load_roi(self, img_size, hook_coords):
    """
    hook_coords: normalized YOLO (x_center, y_center, w, h) in [0,1]
    img_size: (img_height, img_width) or (H, W, C) - we use first two.
    dims: optional crop size control
    returns: x top, y top, roi width, roi height
    """

    # image dims
    img_height = int(img_size[0])
    img_width  = int(img_size[1])

    hook_coords_np = np.asarray(hook_coords, dtype=np.float32).reshape(-1)

    x, y, w, h = hook_coords_np  # normalized
    yb_hook = y + (h / 2.0)

    # features for scaler: [x, y, w, h, yb_hook]
    hook_features = np.array([[x, y, w, h, yb_hook]], dtype=np.float32)
    xs_scaled = feat_scaler.transform(hook_features)  # numpy (1, 5)

    # Get predictions
    xs_t = torch.from_numpy(xs_scaled).float() #.to(device)
    self.model.eval()
    with torch.no_grad():
        pred_scaled = self.model(xs_t).cpu().numpy()

      # pred_inv is [dx, dy]
    pred_inv = target_scaler.inverse_transform(pred_scaled.reshape(1, -1))
    print(pred_inv)
    # dx, dy = pred_inv[0]
    dx, dy = float(pred_inv[0, 0]), float(pred_inv[0, 1])

    # ---- apply offsets relative to hook size (still normalized [0-1]) ----
    x_pred = x + (dx * w)
    y_pred = yb_hook + (dy * h)

    xc = int(round(x_pred * img_width ))
    yc = int(round(y_pred * img_height))

    #crop to top of hook
    gap_pixels = yc - (yb_hook * img_height)

    crop_h = int(gap_pixels * 1.6)
    crop_w = int(crop_h * 1.2)      # Usually loads are wider than they are tall

    xt = int(xc - (crop_w // 2))
    yt = int(yc - (crop_h // 2))

    # Clamp to image bounds
    xt = max(0, min(xt, img_width - crop_w))
    yt = max(0, min(yt, img_height - crop_h))

    return xt, yt, crop_w, crop_h

## Load Classifier Class

In [ ]:
import torch
import albumentations as A
import numpy as np

class Load_Classify:
    def __init__(self, path_to_model):
        self.path = path_to_model
        checkpoint = torch.load(
            self.path,
            map_location='cpu',
            weights_only=False
        )

        self.class_model = checkpoint["model"]
        self.class_model.eval()

        if torch.cuda.is_available():
            self.class_model.cuda()

        self.cls_names = checkpoint["cls_names"]
        self.cls_index = checkpoint["cls_index"]
        self.val_transforms = A.Compose([
            A.Resize(height=240, width=240),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225),
                        max_pixel_value=255.0),
        ])

    def predict_one_image(self, image_array):
        model = self.class_model
        transforms = self.val_transforms

        # model.eval()
        transformed = transforms(image=image_array)['image']
        x = np.transpose(transformed, (2, 0, 1))
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(0)

        if torch.cuda.is_available():
            x = x.cuda()

        with torch.no_grad():
            output = model(x)
            probs = torch.softmax(output, dim=1)
            pred_idx = probs.argmax(dim=1).item()
            confidence = probs[0, pred_idx].item()

        pred_class = self.cls_names[pred_idx]
        return pred_class, confidence

## KYLE Class

In [ ]:
from pydantic import BaseModel

class Box_Result(BaseModel):
    x_c: int
    y_c: int
    width: int
    height: int
    conf: float
    class_type: str

class Image_Result(BaseModel):
    image_name: str
    image_path: str
    image_size: tuple
    image_results: list[Box_Result]

In [ ]:
from ultralytics import YOLO
import cv2
import torch
import numpy as np
from pathlib import Path

def cv_show(name, image):
    cv2.imshow(name, image)
    cv2.waitKey(5000)
    cv2.destroyAllWindows()

class KYLE:
    def __init__(self, objd_model, ob_weights, roi_model, classify_model):
        self.objd_model = objd_model(ob_weights)
        self.objd_model_ss = objd_model(ob_weights)
        self.roi_predict = roi_model
        self.classify_model = classify_model
        self.predictions = []

    def _crop_load(self, img, x, y, w, h):
        return img[y:y+h, x:x+w]

    def _second_stage_detector(self, crop, roi_size):
      """
        From ROI crop predict new object using YOLO
        If object found returns cropped image, local x1 x2, local y1 y2, confidence
        Else - None, 0,0,0,0,0

      """
      print("Predicting Second Crop")
      ss_results = self.objd_model_ss.predict(crop, imgsz=roi_size, stream=False)

      for result in ss_results:
        if not result.boxes:
          continue

        for box in result.boxes.cpu():
            class_id = int(box.cls[0])
            class_name = self.objd_model_ss.names[class_id]  # Get class name using the class ID
            conf = float(box.conf[0])
            label = f"{class_name}: {conf:.2f}"

            if class_id == 1: #Load Class
            #  LOCAL coordinates (relative to the ROI) top left, bottom right
              lx1, ly1, lx2, ly2 = box.xyxy[0].cpu().numpy().astype(int)

              # actual cropped image of the load
              load_only_crop = crop[ly1:ly2, lx1:lx2]
              return load_only_crop, lx1, ly1, lx2, ly2, conf
      return None, 0,0,0,0, 0

    def _predict_from_hook(self, image, box_size, hook_dims):
      """ Input original image, predict the ROI from first pass
          Pass through second stage object detector.
          Classify results
          returns x left, y top, w, h
       """
      print("predict from hook box size and dims",box_size, hook_dims)
      roi = self.roi_predict.predict_load_roi(box_size, hook_dims)
      xl, yt, crop_w, crop_h = roi

      # extract the ROI from the main image
      ROI = image[yt:yt+crop_h, xl:xl+crop_w]
      if ROI.size == 0: return

      # Find the load inside that ROI
      load_crop, lx1, ly1, lx2, ly2, yolo_conf = self._second_stage_detector(ROI, max(crop_w, crop_h))

      if load_crop is not None:
          print("Second ROI detected \n")
          # GLOBAL COORDINATES: ROI start + Local YOLO start
          gx1, gy1 = xl + lx1, yt + ly1
          gx2, gy2 = xl + lx2, yt + ly2

          w, h = gx2 - gx1, gy2 - gy1

          # Draw the final accurate box
          # predicted_class, confidence = self.classify_model.predict_one_image(load_crop)
          # xl, yt, w, h
          print("Second predict", gx1,gy1,w,h)
          return gx1,gy1,w,h
          # return gx1, gy1
      else:
          print("MLP proposed a region, but YOLO stage 2 found no load there. \n")
          # xl, yt, w, h
          return roi

    # Object detection
    def _predict_one_img(self, result, _img, idx):
      load_results = None

      if not result.boxes:
        print("No results")
        image_name = str(getattr(result, "path", f"frame_{idx}")).rsplit("/", 1)[-1]

        return Image_Result(image_name=image_name, image_path=getattr(result, "path", None),
                    image_results=[], image_size=result.orig_shape)


      box_preds = []

      for idx, box in enumerate(result.boxes.cpu()):
        # r = box.xyxy[0].numpy().astype(int) # Get corner points as int (Actual Co-ords)
        class_id = int(box.cls[0])  # Get class ID
        class_name = self.objd_model.names[class_id]  # Get class name using the class ID
        obj_conf = float(box.conf[0])
        label = f"{class_name}: {obj_conf:.2f}"
        # -----------------------------------
        # LOAD (class 1)
        # -----------------------------------
        if class_id == 1:
            xc, yc, w, h = box.xywh[0].cpu().numpy().tolist()
            xc, yc, w, h = int(xc), int(yc), int(w), int(h)
            xl = xc - (w//2)
            yt = yc - (h//2)

            xl = int(xl)
            yt = int(yt)
            w  = int(w)
            h  = int(h)

            # use OD conf for gating
            box_preds.append([1, xc, yc, w, h, obj_conf, "obj", None])

        # -----------------------------------
        # HOOK (class 0)
        # -----------------------------------
        else:
          hook_dims_n = box.xywhn[0]
          hook_dims = box.xywh[0].cpu().tolist()
          print("Hook found - Crop ROI")
          roi_dims = self._predict_from_hook(_img, box.orig_shape, hook_dims_n)
          if roi_dims is None:
            print("ROI is empty")
            continue

          roi_crop = self._crop_load(_img, *roi_dims)

          center = corner_to_center(*roi_dims)
          if isinstance(center, np.ndarray):
              center = center.tolist()
          center = [int(center[0]), int(center[1]), int(center[2]), int(center[3])]
          box_preds.append([0, *hook_dims, None, "hook", idx])
          box_preds.append([1, *center, None, "roi", idx])

      gated_boxes = box_gating(box_preds)
      image_results = []

      for boxes in gated_boxes:
        xc, yc, w, h = boxes[1:5]
        xl,_, yt,_, w, h = center_to_corners(xc, yc, w, h)

        # # clamp
        xl = int(max(0, min(xl, _img.shape[1] - 1)))
        yt = int(max(0, min(yt, _img.shape[0] - 1)))
        w  = int(max(1, min(w, _img.shape[1] - xl)))
        h  = int(max(1, min(h, _img.shape[0] - yt)))

        crop = self._crop_load(_img, xl, yt, w, h)
        pred_class, confidence = self.classify_model.predict_one_image(crop)

        box_result = Box_Result(
              class_type=pred_class,
              x_c=xc,
              y_c=yc,
              width=w,
              height=h,
              conf=confidence,
              )
        image_results.append(box_result)

      image_name = result.path.rsplit("/", 1)[-1]
      load_results = Image_Result(image_name=image_name, image_path = result.path,image_results = image_results, image_size=result.orig_shape)
      return load_results

    #Run inference
    def detect(self, input):
        results = self.objd_model.predict(input, stream=False)
        if results is None:
          print("No predictions")
          return

        crop_results = None

        if isinstance(self.objd_model, YOLO):
            outputs = []
            for r in results:
                img = r.orig_img
                out = self._predict_one_img(r, img, idx=1)
                outputs.append(out)
                self.predictions.append(out)
            return outputs

    # #Run inference on bulk images - upload array of images.
    def detect_stream(self, stream):
        results = self.objd_model.predict(stream, stream=True)

        for i, r in enumerate(results):
            img = r.orig_img
            out = self._predict_one_img(r, img, idx=i)
            self.predictions.append(out)
            yield out

## Run KYLE Model

In [ ]:
best_weights = "/content/drive/MyDrive/Crane-Detector/runs/detect/best_260126_train7/weights/best.pt"
yolo_model = YOLO(best_weights)

ROI_PATH = "/content/drive/MyDrive/Crane-Detector/LoadRegressionModel/LoadRegressionStandardv2.pth"
roi_predict = Roi_Predict(ROI_PATH)

CLS_PATH = "/content/drive/MyDrive/Crane-Detector/Image_Classes/crane_classifier.pth"
class_model = Load_Classify(CLS_PATH)

In [ ]:
kyle = KYLE(YOLO, best_weights, roi_predict, class_model);

In [ ]:
img = cv2.imread('/content/drive/MyDrive/Crane-Detector/First Experiment/All_2LCS_AUG/test/images/2505301040060000_jpg.rf.e1ed3ff61e40b48bd7aecdf5f7df6050.jpg') #1 hook, 1 load
# img = '/content/drive/MyDrive/Crane-Detector/First Experiment/All_2LCS_AUG/test/images/2505301040060000_jpg.rf.e1ed3ff61e40b48bd7aecdf5f7df6050.jpg' #1 hook, 1 load

In [ ]:
cv2_imshow(img)

In [ ]:
kyle.detect(img)

In [ ]:
kyle.predictions

#Test KYLE Model

In [ ]:
test_path = Path("/content/TEST_ONLY-2/test")

test_image_path = test_path / "images"
test_labels_path = test_path / "labels"

test_images = [img for img in test_image_path.glob("*.jpg")]
test_labels = [img for img in test_labels_path.glob("*.txt")]

In [ ]:
type(test_images[0]), test_labels[0]

In [ ]:
with open(test_labels[0], "r") as f:
  coords = f.readlines()
  print(extract_coords(coords))

In [ ]:
import yaml

def parse_yaml(PATH):
  cls_names = []
  test_path = None
  train_path = None
  valid_path = None

  with open(PATH) as stream:
      try:
          yamlLoad = yaml.safe_load(stream)
          cls_names = yamlLoad['names']
          test_path = yamlLoad['test']
          train_path = yamlLoad['train']
          valid_path = yamlLoad['val']
      except yaml.YAMLError as exc:
          print(exc)
  return cls_names, test_path, train_path, valid_path


In [ ]:
cls_names = parse_yaml("/content/TEST_ONLY-2/data.yaml"); cls_names

In [ ]:
kyle = KYLE(YOLO, best_weights, roi_predict, class_model);

In [ ]:
# test_img = cv2.imread()
# results = kyle.detect_stream([img, img, img])

In [ ]:
# next(results)

In [ ]:
test_images = ["content/TEST_ONLY-1/test/images/2503111210060000_jpg.rf.921ebef6d4caaaa25256480e6aaba8db.jpg", "/content/TEST_ONLY-1/test/images/2503111210060000_jpg.rf.921ebef6d4caaaa25256480e6aaba8db.jpg"]

In [ ]:
# res = kyle.detect_stream(test_img)

In [ ]:
# kyle.predictions

In [ ]:
# test_img = cv2.imread("/content/TEST_ONLY-1/test/images/2503111210060000_jpg.rf.921ebef6d4caaaa25256480e6aaba8db.jpg")

In [ ]:
def show_image_bb(im, dims):
  pt1 = (int(dims[0]), int(dims[2])) #top left corner
  pt2 = (int(dims[1]), int(dims[3])) #bottom right corner

  cv2.rectangle(im, pt1, pt2, (0, 255, 0), 2)
  cv2_imshow(im)

In [ ]:
def IOU(gt, pred):
  x_min = max(gt[0], pred[0])
  x_max = min(gt[1], pred[1])
  y_min = max(gt[2], pred[2])
  y_max = min(gt[3], pred[3])

  area_gt = (gt[4] * gt[5])
  area_pred = (pred[4] * pred[5])

  iw = max(0.0, x_max - x_min)
  ih = max(0.0, y_max - y_min)
  intersection = iw * ih

  union = area_gt + area_pred - intersection

  iou = intersection / union

  # print(x_min, x_max, y_min, y_max, area_gt, area_pred)
  # print("intersection: ", intersection)
  # print("union: ", union)
  # print("iou: ", iou)
  return iou

In [ ]:
def get_test_pairs(gt_labels, predict_labels, image_size):
  """
  Takes ground_truth normalised labels [list] and actual box predictions as a list[box_preds]
  Check if gt box fits in prediction box using UOI
  if multiple return highest UOI - if multiple at 100% return closest center
  return [gt_index, gt_class, predict_index, predict_cls, conf]
  """
  pairs = []

  for i, label in enumerate(gt_labels):
    gt_cls = int(label[0])
    gt_xc = label[1] * image_size[1]
    gt_yc = label[2]* image_size[0]
    gt_w = label[3] * image_size[1]
    gt_h = label[4]* image_size[0]

    gt_dims = center_to_corners(gt_xc, gt_yc, gt_w, gt_h)

    for j, pred_labels in enumerate(predict_labels):
      pred_cls = pred_labels.class_type
      pred_xc = pred_labels.x_c
      pred_yc = pred_labels.y_c
      pred_w = pred_labels.width
      pred_h = pred_labels.height

      pred_dims = center_to_corners(pred_xc, pred_yc,pred_w, pred_h)

      iou = IOU(gt_dims, pred_dims)
      if iou > 0:
        pairs.append([i, j, iou])

  print(pairs)
  return pairs

In [ ]:
cls_names[0]

In [ ]:
cls_to_index = {cls.lower():i for i, cls in enumerate(cls_names[0])}; cls_to_index.get('rebar')

In [ ]:
import pandas as pd

def test_preds(PATH):
  yamlLoad = parse_yaml(PATH + "/data.yaml")
  cls_names, _, _, _ = yamlLoad
  cls_index = {cls.lower():i for i, cls in enumerate(cls_names)}
  print(cls_index)
  test_path = Path(PATH+"/test")
  test_image_path = (test_path / "images")
  test_labels_path = (test_path / "labels")

  test_images = [img for img in test_image_path.glob("*.jpg")] #********** uncomment when testing full suite
  # test_images = [Path("/content/TEST_ONLY-1/test/images/2503111210060000_jpg.rf.921ebef6d4caaaa25256480e6aaba8db.jpg")]
  predict = kyle.detect_stream(test_images)

  results = pd.DataFrame(columns = ["image","image_size" , "gt_index", "gt_class", "pred_index", "pred_class", "conf", "xc", "yc", "w", "h"])
  pairs = []

  # for _ in predict:
  #   pass

  for i, img in enumerate(predict):
    print(f"i=, img=")
    image_name = img.image_name.rsplit(".", 1)[0]
    image_label_path = test_labels_path / (image_name + ".txt")
    image_size = img.image_size

    gt_label_dims = None

    with open(image_label_path, "r") as f:
      lines = f.readlines()
      gt_label_dims = extract_coords(lines)

    box_results = img.image_results

    # Match all gt_labels to all box_results using intersection of union for overlapping boxes
    pairs = get_test_pairs(gt_label_dims, box_results, image_size)

    if not pairs:
      result = {"image": image_name, "image_size": image_size, "gt_index":None, "gt_class": None, "pred_index":None , "pred_class": None,
                "conf":None, "xc":None,"yc":None,"w":None,"h":None}
      results.loc[len(results)] = result
      continue


    # gt_idx, pred_idx, _ = pairs

    for pair in pairs:
      gt_idx, pred_idx, _ = pair

      gt_cls = gt_label_dims[gt_idx][0]
      pred_cls = box_results[pred_idx].class_type
      pred_cls_idx = cls_index.get(pred_cls.lower())
      pred_conf = box_results[pred_idx].conf
      pred_x = box_results[pred_idx].x_c
      pred_y = box_results[pred_idx].y_c
      pred_w = box_results[pred_idx].width
      pred_h = box_results[pred_idx].height

      result = {"image": image_name, "image_size": image_size, "gt_index":gt_idx, "gt_class": gt_cls, "pred_index":pred_idx , "pred_class": pred_cls_idx,
                "conf":pred_conf, "xc":pred_x,"yc":pred_y,"w":pred_w,"h":pred_h}
      results.loc[len(results)] = result


  return results
      # print(image_name, lines, "\n")
    # test_labels = [img for img in test_labels_path.glob(image_name+".txt")]

In [ ]:
kyle = KYLE(YOLO, best_weights, roi_predict, class_model);


In [ ]:
results = test_preds("/content/TEST_ONLY-2")

In [ ]:
results

In [ ]:
# results['gt_class'] = results['gt_class'].replace({None: -1, np.nan: -1}).astype(int)

In [ ]:
results['true'] = np.where(results['gt_class']== results['pred_class']
                     , True, False)

In [ ]:
results

In [ ]:
# To do - calculate true and false
# Print image and labels to cv2  show_image_bb(test_img, gt) for true ones

In [ ]:
results.true.value_counts()

In [ ]:
filepath = Path("/content/drive/MyDrive/Crane-Detector/YOLOv8 Normal/out.csv")
filepath.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(filepath)

In [ ]:
image_list = results.loc[results['true'] == True, ['image','xc','yc',	'w',	'h', 'pred_class', 'image_size']].values.tolist()

In [ ]:
image_list

In [ ]:
# for images in image_list:
#   path = "/content/TEST_ONLY-2/test/images/"
#   img_path = f"{path}/{images[0]}.jpg"
#   img = cv2.imread(img_path)
#   cls_idx = images[5]
#   cls = cls_names[0][cls_idx]
#   print(cls)
#   corners = center_to_corners(*images[1:5])
#   print(corners)
#   show_image_bb(img, corners)

## Further Improvement - Notes
  - Bounding Boxes
    - Far too big
  - Second predictor
    - Train a model on smaller crops?
  - Classification isn't great, 60% in training and doesnt performs too great in this model.
    Suggestions:
      - increase number of training data (More manual crops from lobster cam) and better quality images.
      - Include a not class (Background / Not a load)
      - Play around with transforms and augmentations
      - Increase model architecture.
      - TTA
  - Finish the Muli-Res YOLO model.